# 02b - Auto-Annotation with Grounding DINO
## Bradford Bulls - AI Sponsorship Exposure Valuation System

**Problem:** 6000+ frames is too many to annotate manually.

**Solution:** 
1. Select ~500-800 most diverse frames (skip near-duplicates)
2. Use **Grounding DINO** (zero-shot object detection) to auto-detect logos by text prompt
3. Generate YOLO-format annotation files
4. Upload images + annotations to Roboflow → only need to **review & correct**

This reduces manual work from drawing 6000+ bounding boxes to just reviewing/fixing pre-drawn ones.

## 1. Install Dependencies

In [1]:
# Install autodistill + Grounding DINO (one-time)
# autodistill uses foundation models to auto-label datasets
# !pip install autodistill autodistill-grounding-dino supervision -q

In [2]:
import sys
sys.path.insert(0, "..")

import cv2
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from tqdm import tqdm

from src.config import (
    FRAMES_CLEAR_DIR, METADATA_DIR, OUTPUT_DIR,
    SPONSOR_LABELS, DEVICE
)

print(f"Device: {DEVICE}")
print(f"Frames available: {len(list(FRAMES_CLEAR_DIR.glob('frame_*.jpg')))}")

[Device] Using: mps (Apple Silicon (Metal))
Device: mps
Frames available: 6482


## 2. Select Diverse Frame Subset

From 6000+ frames, select ~500-800 most diverse frames using visual similarity filtering.
This avoids annotating near-identical frames and gives better training data diversity.

In [3]:
import imagehash
from PIL import Image

# ============================================================
# CONFIG: How many frames to select for annotation
# ============================================================
TARGET_ANNOTATION_COUNT = 600  # Adjust as needed (500-800 recommended)
DIVERSITY_HASH_THRESHOLD = 8  # Higher = more similar frames allowed

all_frames = sorted(FRAMES_CLEAR_DIR.glob("frame_*.jpg"))
print(f"Total clear frames: {len(all_frames)}")

# Strategy: Walk through frames, keep only those that are visually different
# from the last kept frame (using perceptual hash)
selected_frames = []
last_hash = None

for frame_path in tqdm(all_frames, desc="Selecting diverse frames"):
    img = Image.open(frame_path)
    current_hash = imagehash.phash(img)
    
    if last_hash is None or (current_hash - last_hash) >= DIVERSITY_HASH_THRESHOLD:
        selected_frames.append(frame_path)
        last_hash = current_hash

print(f"\nAfter diversity filtering: {len(selected_frames)} frames")

# If still too many, subsample evenly
if len(selected_frames) > TARGET_ANNOTATION_COUNT:
    step = len(selected_frames) / TARGET_ANNOTATION_COUNT
    indices = [int(i * step) for i in range(TARGET_ANNOTATION_COUNT)]
    selected_frames = [selected_frames[i] for i in indices]
    print(f"After subsampling: {len(selected_frames)} frames")

print(f"\nSelected {len(selected_frames)} frames for auto-annotation")

Total clear frames: 6482


Selecting diverse frames: 100%|██████████| 6482/6482 [00:20<00:00, 320.73it/s]


After diversity filtering: 6164 frames
After subsampling: 600 frames

Selected 600 frames for auto-annotation


In [4]:
# Copy selected frames to a separate directory for annotation
ANNOTATION_DIR = OUTPUT_DIR / "frames_for_annotation"
ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)

for fp in tqdm(selected_frames, desc="Copying selected frames"):
    shutil.copy2(fp, ANNOTATION_DIR / fp.name)

print(f"Copied {len(selected_frames)} frames to: {ANNOTATION_DIR}")

Copying selected frames: 100%|██████████| 600/600 [00:00<00:00, 2600.42it/s]

Copied 600 frames to: /Users/hoamai/Bradford/BRADFORD_BULLS_PROJECT/notebooks/../output/frames_for_annotation


## 3. Auto-Annotate with Grounding DINO

Grounding DINO is a **zero-shot** object detector — it can find objects by text description without any training.

We use text prompts like `"AON logo"`, `"sponsor logo on jersey"`, `"text on shirt"` to detect logo regions automatically.

In [5]:
from autodistill_grounding_dino import GroundingDINO
from autodistill.detection import CaptionOntology

# ============================================================
# ONTOLOGY: Map text prompts → label classes
# ============================================================
# Grounding DINO uses text prompts to detect objects.
# We map each prompt to our sponsor label.
# 
# Strategy: 
#   - Use generic prompts for broad detection ("logo", "sponsor text")
#   - Use specific prompts for distinctive logos ("AON", "MCP")
#   - Grounding DINO will find bounding boxes, we assign labels
# ============================================================

ontology = CaptionOntology({
    # Specific brand logos (high confidence matches)
    "AON logo": "aon",
    "AON text on jersey": "aon",
    "MCP logo": "mcp",
    "MCP text on jersey": "mcp",
    "Cedar Court Hotels logo": "cch_cedar_court",
    "ChadLaw logo": "chadlaw",
    "ATM Hospitality logo": "atm_hospitality",
    "EM workwear logo": "em_workwear",
    "Fairway Flooring logo": "fairway_flooring",
    "KLG logo": "klg",
    "MNA logo": "mna_cladding",
    "MNA text on shorts": "mna_support",
    "Bartercard logo": "bartercard",
    "Top Notch logo": "top_notch",
    "Romantica logo": "romantica_beds",
    
    # Generic fallback prompts
    "sponsor logo on rugby jersey": "sponsor_logo",
    "text on rugby shirt": "sponsor_logo",
    "brand logo on sportswear": "sponsor_logo",
})

print("Ontology configured with", len(ontology.promptMap), "prompts")
print("\nPrompt → Label mappings:")
for prompt, label in ontology.promptMap:
    print(f"  '{prompt}' → {label}")

Ontology configured with 18 prompts

Prompt → Label mappings:
  'AON logo' → aon
  'AON text on jersey' → aon
  'MCP logo' → mcp
  'MCP text on jersey' → mcp
  'Cedar Court Hotels logo' → cch_cedar_court
  'ChadLaw logo' → chadlaw
  'ATM Hospitality logo' → atm_hospitality
  'EM workwear logo' → em_workwear
  'Fairway Flooring logo' → fairway_flooring
  'KLG logo' → klg
  'MNA logo' → mna_cladding
  'MNA text on shorts' → mna_support
  'Bartercard logo' → bartercard
  'Top Notch logo' → top_notch
  'Romantica logo' → romantica_beds
  'sponsor logo on rugby jersey' → sponsor_logo
  'text on rugby shirt' → sponsor_logo
  'brand logo on sportswear' → sponsor_logo


Importing from timm.models.layers is deprecated, please import via timm.layers


In [6]:
# ============================================================
# RUN AUTO-ANNOTATION
# ============================================================
# This will process all selected frames and generate YOLO format labels.
# On MacBook (MPS): ~1-3 seconds per frame
# On GPU (CUDA):    ~0.2-0.5 seconds per frame

AUTOLABEL_OUTPUT = OUTPUT_DIR / "auto_annotated"
AUTOLABEL_OUTPUT.mkdir(parents=True, exist_ok=True)

base_model = GroundingDINO(
    ontology=ontology,
    box_threshold=0.2,     # Lower = more detections (but more false positives)
    text_threshold=0.15,   # Lower = more sensitive to text matches
)

# Run labeling on the selected frames directory
base_model.label(
    input_folder=str(ANNOTATION_DIR),
    output_folder=str(AUTOLABEL_OUTPUT),
)

print(f"\nAuto-annotation complete!")
print(f"Output: {AUTOLABEL_OUTPUT}")

trying to load grounding dino directly
downloading dino model weights


torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/TensorShape.cpp:4383.)


final text_encoder_type: bert-base-uncased


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


AttributeError: 'BertModel' object has no attribute 'get_head_mask'

## 4. Preview Auto-Annotations

Visualize some auto-annotated frames to check quality before uploading to Roboflow.

In [ ]:
# Build class name list from ontology
class_names = list(dict.fromkeys(ontology.promptMap.values()))  # unique, ordered
print(f"Class names ({len(class_names)}): {class_names}\n")

def visualize_annotation(image_path, label_path, class_names):
    """Draw YOLO bounding boxes on image."""
    img = cv2.imread(str(image_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    fig, ax = plt.subplots(1, 1, figsize=(14, 8))
    ax.imshow(img_rgb)
    
    if label_path.exists():
        with open(label_path) as f:
            lines = f.readlines()
        
        colors = plt.cm.Set3(np.linspace(0, 1, len(class_names)))
        
        for line in lines:
            parts = line.strip().split()
            cls_id = int(parts[0])
            cx, cy, bw, bh = [float(x) for x in parts[1:5]]
            
            # Convert YOLO format (center, w, h) to pixel coordinates
            x1 = (cx - bw/2) * w
            y1 = (cy - bh/2) * h
            box_w = bw * w
            box_h = bh * h
            
            label = class_names[cls_id] if cls_id < len(class_names) else f"class_{cls_id}"
            color = colors[cls_id % len(colors)]
            
            rect = patches.Rectangle(
                (x1, y1), box_w, box_h,
                linewidth=2, edgecolor=color, facecolor='none'
            )
            ax.add_patch(rect)
            ax.text(x1, y1 - 5, label, color=color, fontsize=10,
                    fontweight='bold', backgroundcolor='black')
        
        ax.set_title(f"{image_path.name} — {len(lines)} detections", fontsize=11)
    else:
        ax.set_title(f"{image_path.name} — No detections", fontsize=11)
    
    ax.axis("off")
    plt.tight_layout()
    plt.show()

# Preview 8 annotated frames
annotated_images = sorted((AUTOLABEL_OUTPUT / "images").glob("*.jpg"))
annotated_labels = AUTOLABEL_OUTPUT / "labels"

if not annotated_images:
    annotated_images = sorted((AUTOLABEL_OUTPUT / "train" / "images").glob("*.jpg"))
    annotated_labels = AUTOLABEL_OUTPUT / "train" / "labels"

print(f"Total auto-annotated images: {len(annotated_images)}")

# Show a sample
sample_step = max(1, len(annotated_images) // 8)
for i in range(0, min(len(annotated_images), 8 * sample_step), sample_step):
    img_path = annotated_images[i]
    lbl_path = annotated_labels / (img_path.stem + ".txt")
    visualize_annotation(img_path, lbl_path, class_names)

## 5. Annotation Statistics

In [ ]:
# Count detections per class
from collections import Counter

label_files = sorted(annotated_labels.glob("*.txt"))
class_counts = Counter()
total_boxes = 0
frames_with_detections = 0

for lf in label_files:
    lines = lf.read_text().strip().split("\n")
    if lines and lines[0]:
        frames_with_detections += 1
        for line in lines:
            cls_id = int(line.split()[0])
            label = class_names[cls_id] if cls_id < len(class_names) else f"class_{cls_id}"
            class_counts[label] += 1
            total_boxes += 1

print("=" * 50)
print("AUTO-ANNOTATION STATISTICS")
print("=" * 50)
print(f"Total frames processed:       {len(annotated_images)}")
print(f"Frames with detections:       {frames_with_detections}")
print(f"Frames without detections:    {len(annotated_images) - frames_with_detections}")
print(f"Total bounding boxes:         {total_boxes}")
print(f"\nDetections per class:")
for label, count in class_counts.most_common():
    print(f"  {label:<25s} {count:>5}")
print("=" * 50)

## 6. Upload Auto-Annotated Data to Roboflow

Upload images **with pre-drawn annotations** to Roboflow.
On Roboflow, you only need to **review and correct** — not draw from scratch.

In [ ]:
# ============================================================
# ROBOFLOW CONFIG
# ============================================================
ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"
PROJECT_NAME = "kit-sponsor-logos"

from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
workspace = rf.workspace()

try:
    project = workspace.project(PROJECT_NAME)
    print(f"Connected to: {project.name}")
except Exception:
    project = workspace.create_project(
        project_name="Kit Sponsor Logos",
        project_type="object-detection",
        project_license="MIT",
        annotation="kit-sponsor-logos",
    )
    print(f"Created project: Kit Sponsor Logos")

In [ ]:
# Upload images WITH annotations to Roboflow
# Roboflow accepts YOLO-format .txt annotation files alongside images

success = 0
errors = 0

for img_path in tqdm(annotated_images, desc="Uploading with annotations"):
    label_path = annotated_labels / (img_path.stem + ".txt")
    
    try:
        if label_path.exists() and label_path.read_text().strip():
            # Upload image WITH annotation
            project.upload(
                image_path=str(img_path),
                annotation_path=str(label_path),
                split="train",
            )
        else:
            # Upload image without annotation (will need manual labeling)
            project.upload(
                image_path=str(img_path),
                split="train",
            )
        success += 1
    except Exception as e:
        errors += 1
        if errors <= 3:
            print(f"  Error: {img_path.name}: {e}")

print(f"\nUpload complete!")
print(f"  Success: {success}")
print(f"  Errors:  {errors}")
print(f"\nNext steps:")
print(f"  1. Go to Roboflow → Annotate tab")
print(f"  2. Review and correct the pre-drawn bounding boxes")
print(f"  3. Fix any wrong labels or missed logos")
print(f"  4. Generate a dataset version when done")

## 7. Alternative: Quick Manual Seed → Roboflow Auto-Label

If Grounding DINO results are not accurate enough, try this faster workflow:

1. **Manually annotate just 30-50 frames** on Roboflow (highest quality)
2. **Generate a version** with those 30-50 frames
3. **Train a model** on Roboflow (they offer free training)
4. Use Roboflow's **Label Assist** to auto-label remaining frames with YOUR trained model
5. Review and correct

This is often more accurate than Grounding DINO because the model learns YOUR specific logos.